# Tiny CLIP: compositional color-shape holdouts

**Question:** Can a tiny CLIP-style model align recombinable visual and language attributes when complete color-shape pairs are held out?

The reusable implementation lives in `src/clip_repro`. This notebook owns only the experiment configuration, execution, and interpretation.

## Prediction and controls

- Training and fresh-seen retrieval should approach 100%.
- Successful compositional generalization requires positive color and shape margins on held-out pairs.
- Training uses one instance of every unique semantic combination per batch, preventing duplicate-caption false negatives.
- Evaluation rendering seeds are fixed across training seeds so model variation is not mixed with test-sample variation.

In [1]:
from collections import Counter

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms.functional import pil_to_tensor

from clip_repro.data import (
    ShapeDataset,
    all_combinations,
    build_splits,
    build_tokenizer,
    make_collate_fn,
    render_shape,
)
from clip_repro.evaluation import evaluate_retrieval, prediction_counts, summarize_pair
from clip_repro.train import TrainConfig, train_model
from clip_repro.utils import get_device, set_seed


In [2]:
HELD_OUT_PAIRS = {
    ("red", "triangle"),
    ("blue", "square"),
}
TRAIN_SEEDS = [0, 1, 2, 3, 4]
PROBE_MODEL_SEED = 2  # Deliberately selected clear failure case; not a population estimate.
HELD_OUT_EVAL_SEED = 12345
SEEN_EVAL_SEED = 54321
SAMPLES_PER_COMBO = 20
TRAIN_CONFIG = TrainConfig(epochs=500, learning_rate=3e-4)
device = get_device()

In [3]:
train_combinations, test_combinations = build_splits(HELD_OUT_PAIRS)
candidate_combinations = all_combinations()
tokenizer = build_tokenizer(candidate_combinations)

train_loader = DataLoader(
    ShapeDataset(train_combinations),
    batch_size=len(train_combinations),
    shuffle=True,
    collate_fn=make_collate_fn(tokenizer),
)

assert len(train_combinations) == 42
assert len(test_combinations) == 12
len(train_combinations), len(test_combinations), tokenizer.vocab_size

(42, 12, 16)

## Single-seed baseline

In [4]:
model, history = train_model(
    tokenizer,
    train_loader,
    device,
    TRAIN_CONFIG,
    seed=0,
    print_every=25,
)
pd.DataFrame(history).tail(1)

Epoch    1 | Loss 3.8774 | I2T 0.024 | T2I 0.024 | tau 0.0700
Epoch   25 | Loss 1.1778 | I2T 0.548 | T2I 0.619 | tau 0.0699
Epoch   50 | Loss 0.7359 | I2T 0.738 | T2I 0.738 | tau 0.0694
Epoch   75 | Loss 0.4165 | I2T 0.881 | T2I 0.929 | tau 0.0689
Epoch  100 | Loss 0.2802 | I2T 0.952 | T2I 0.976 | tau 0.0685
Epoch  125 | Loss 0.1339 | I2T 1.000 | T2I 1.000 | tau 0.0681
Epoch  150 | Loss 0.0820 | I2T 1.000 | T2I 1.000 | tau 0.0678
Epoch  175 | Loss 0.0508 | I2T 1.000 | T2I 1.000 | tau 0.0675
Epoch  200 | Loss 0.0374 | I2T 1.000 | T2I 1.000 | tau 0.0673
Epoch  225 | Loss 0.0307 | I2T 1.000 | T2I 1.000 | tau 0.0671
Epoch  250 | Loss 0.0278 | I2T 1.000 | T2I 1.000 | tau 0.0669
Epoch  275 | Loss 0.0248 | I2T 1.000 | T2I 1.000 | tau 0.0667
Epoch  300 | Loss 0.0226 | I2T 1.000 | T2I 1.000 | tau 0.0665
Epoch  325 | Loss 0.0190 | I2T 1.000 | T2I 1.000 | tau 0.0664
Epoch  350 | Loss 0.0178 | I2T 1.000 | T2I 1.000 | tau 0.0662
Epoch  375 | Loss 0.0165 | I2T 1.000 | T2I 1.000 | tau 0.0660
Epoch  4

,epoch,loss,i2t,t2i,temperature
499,500,0.012773,1.0,1.0,0.065319


In [5]:
set_seed(HELD_OUT_EVAL_SEED)
held = evaluate_retrieval(
    model, test_combinations, candidate_combinations, tokenizer, device, SAMPLES_PER_COMBO
)

set_seed(SEEN_EVAL_SEED)
seen = evaluate_retrieval(
    model, train_combinations, candidate_combinations, tokenizer, device, SAMPLES_PER_COMBO
)

{
    "fresh_seen_accuracy": seen["accuracy"],
    "held_out_accuracy": held["accuracy"],
    "held_out_color_margin": held["color_margin"],
    "held_out_shape_margin": held["shape_margin"],
}

{'fresh_seen_accuracy': 1,
 'held_out_accuracy': 0.7166666666666667,
 'held_out_color_margin': 0.3112184832493464,
 'held_out_shape_margin': 0.11975961253046989}

In [6]:
pair_summaries = {pair: summarize_pair(held, pair) for pair in HELD_OUT_PAIRS}
pair_summaries

{('blue', 'square'): {'accuracy': 0.43333333333333335,
  'color_margin': 0.36594423552354177,
  'shape_margin': -0.006034561991691589},
 ('red', 'triangle'): {'accuracy': 1,
  'color_margin': 0.2564927309751511,
  'shape_margin': 0.24555378705263137}}

In [7]:
for pair in HELD_OUT_PAIRS:
    print(f"\n{pair}")
    for caption, count in prediction_counts(held, pair).most_common(8):
        print(count, caption)


('blue', 'square')
19 a large blue square on the left
16 a large blue circle in the center
16 a large blue circle on the right
13 a small blue circle on the right
11 a small blue circle on the left
11 a small blue circle in the center
9 a small blue square on the left
9 a small blue square in the center

('red', 'triangle')
20 a small red triangle on the left
20 a small red triangle in the center
20 a small red triangle on the right
20 a large red triangle on the left
20 a large red triangle in the center
20 a large red triangle on the right


## Five-seed replication

In [8]:
rows = []
probe_model = None

for seed in TRAIN_SEEDS:
    model, history = train_model(
        tokenizer, train_loader, device, TRAIN_CONFIG, seed=seed
    )

    set_seed(HELD_OUT_EVAL_SEED)
    held = evaluate_retrieval(
        model, test_combinations, candidate_combinations, tokenizer, device, SAMPLES_PER_COMBO
    )
    set_seed(SEEN_EVAL_SEED)
    seen = evaluate_retrieval(
        model, train_combinations, candidate_combinations, tokenizer, device, SAMPLES_PER_COMBO
    )

    row = {
        "seed": seed,
        "train_loss": history[-1]["loss"],
        "seen_accuracy": seen["accuracy"],
        "held_accuracy": held["accuracy"],
        "held_color_margin": held["color_margin"],
        "held_shape_margin": held["shape_margin"],
    }
    for pair in HELD_OUT_PAIRS:
        name = "_".join(pair)
        summary = summarize_pair(held, pair)
        row.update({f"{name}_{metric}": value for metric, value in summary.items()})
    rows.append(row)

    if seed == PROBE_MODEL_SEED:
        probe_model = model

assert probe_model is not None
results = pd.DataFrame(rows)
results


,seed,train_loss,seen_accuracy,held_accuracy,held_color_margin,held_shape_margin,blue_square_accuracy,blue_square_color_margin,blue_square_shape_margin,red_triangle_accuracy,red_triangle_color_margin,red_triangle_shape_margin
0,0,0.012772,1,0.716667,0.311410,0.119054,0.433333,0.366043,-0.006499,1.000000,0.256778,0.244606
1,1,0.013171,1,0.491667,0.254941,0.013240,0.000000,0.313337,-0.165485,0.983333,0.196545,0.191965
2,2,0.014921,1,0.500000,0.369034,-0.020492,0.000000,0.446902,-0.166131,1.000000,0.291165,0.125147
3,3,0.014431,1,0.500000,0.305392,-0.038854,0.000000,0.368639,-0.286102,1.000000,0.242146,0.208394
4,4,0.013112,1,0.512500,0.302973,0.075428,0.025000,0.356045,-0.135023,1.000000,0.249900,0.285880


In [9]:
results.agg(["mean", "std"])

,seed,train_loss,seen_accuracy,held_accuracy,held_color_margin,held_shape_margin,blue_square_accuracy,blue_square_color_margin,blue_square_shape_margin,red_triangle_accuracy,red_triangle_color_margin,red_triangle_shape_margin
mean,2.000000,0.013681,1.0,0.544167,0.308750,0.029675,0.091667,0.370193,-0.151848,0.996667,0.247307,0.211198
std,1.581139,0.000937,0.0,0.096717,0.040562,0.066267,0.191304,0.048313,0.099779,0.007454,0.033998,0.060158


## Result record before refactor

For the `red-triangle / blue-square` holdout, five earlier runs produced 100% fresh-seen accuracy. Red-triangle shape retrieval was nearly perfect (mean 99.3%, mean shape margin +0.208), while blue-square retrieval was poor (mean 9.2%, mean shape margin -0.151) and usually mapped to blue-circle captions.

**Current interpretation:** the failure is pair-specific rather than a general inability to perceive shape. The next discriminating experiment should localize whether square information remains linearly decodable before and after the CLIP image projection.

## Frozen linear probe: where does blue-square shape information disappear?

**Question:** When CLIP retrieval maps a held-out blue square to a blue-circle caption, is square information still linearly decodable from the frozen image pathway?

We compare:

1. flattened CNN features immediately before `image_encoder.projection`, and
2. the final normalized projected image embedding used by CLIP retrieval.

The probe is a single linear layer trained only on fresh renderings of the 42 seen combinations. The CLIP model remains frozen. Evaluation uses the existing fixed rendering seeds and reports red-triangle and blue-square separately.

**Committed prediction:** blue-square shape will remain linearly decodable from the final projected embedding. If retrieval still fails, that would favor a cross-modal alignment explanation over loss of visual shape information.

**Controls:** fresh-seen probe accuracy must be high; renderings are identical across stages; probe loss is class-balanced because the seen split contains more circle combinations than square or triangle combinations.


In [10]:
# Verify that the selected frozen model exhibits the retrieval failure we want to localize.
probe_model.eval()
for parameter in probe_model.parameters():
    parameter.requires_grad_(False)

assert not any(parameter.requires_grad for parameter in probe_model.parameters())

set_seed(HELD_OUT_EVAL_SEED)
probe_retrieval = evaluate_retrieval(
    probe_model,
    test_combinations,
    candidate_combinations,
    tokenizer,
    device,
    SAMPLES_PER_COMBO,
)

{
    pair: {
        **summarize_pair(probe_retrieval, pair),
        "top_predictions": prediction_counts(probe_retrieval, pair).most_common(3),
    }
    for pair in sorted(HELD_OUT_PAIRS)
}


{('blue', 'square'): {'accuracy': 0,
  'color_margin': 0.44690194522651533,
  'shape_margin': -0.16613146414359412,
  'top_predictions': [('a small blue circle on the left', 20),
   ('a small blue circle in the center', 20),
   ('a small blue circle on the right', 20)]},
 ('red', 'triangle'): {'accuracy': 1,
  'color_margin': 0.29116520980993904,
  'shape_margin': 0.12514717976252238,
  'top_predictions': [('a small red triangle on the left', 20),
   ('a small red triangle in the center', 20),
   ('a small red triangle on the right', 20)]}}

In [11]:
SHAPE_TO_ID = {"circle": 0, "triangle": 1, "square": 2}
ID_TO_SHAPE = {index: shape for shape, index in SHAPE_TO_ID.items()}

PROBE_TRAIN_RENDER_SEED = 111
PROBE_INITIALIZATION_SEED = 222
PROBE_SAMPLES_PER_COMBO = 100
PROBE_EPOCHS = 100


@torch.no_grad()
def make_probe_feature_sets(
    model,
    combinations,
    samples_per_combo,
    render_seed,
    batch_size=256,
):
    """Render once and extract both stages from exactly the same images."""
    set_seed(render_seed)
    model.eval()

    feature_chunks = {"pre_projection": [], "projected": []}
    labels = []
    sample_combinations = []
    image_batch = []

    def encode_pending_images():
        if not image_batch:
            return

        images = torch.stack(image_batch).to(device)
        pre_projection = model.image_encoder.features(images).flatten(start_dim=1)
        projected = F.normalize(
            model.image_encoder.projection(pre_projection),
            dim=-1,
        )

        feature_chunks["pre_projection"].append(pre_projection.cpu())
        feature_chunks["projected"].append(projected.cpu())
        image_batch.clear()

    for combo in combinations:
        for _ in range(samples_per_combo):
            image = pil_to_tensor(render_shape(combo)).float().div(255.0)
            image_batch.append(image)
            labels.append(SHAPE_TO_ID[combo[1]])
            sample_combinations.append(combo)

            if len(image_batch) == batch_size:
                encode_pending_images()

    encode_pending_images()

    features = {
        stage: torch.cat(chunks)
        for stage, chunks in feature_chunks.items()
    }
    return features, torch.tensor(labels), sample_combinations


In [12]:
def train_linear_probe(
    features,
    labels,
    seed=PROBE_INITIALIZATION_SEED,
    epochs=PROBE_EPOCHS,
    learning_rate=1e-2,
    batch_size=256,
):
    set_seed(seed)
    probe = nn.Linear(features.shape[1], len(SHAPE_TO_ID)).to(device)

    # The holdout makes circle more frequent than triangle or square in probe training.
    # Inverse-frequency weights prevent the probe from winning by preferring circle.
    class_counts = torch.bincount(labels, minlength=len(SHAPE_TO_ID)).float()
    class_weights = class_counts.sum() / class_counts
    class_weights = (class_weights / class_weights.mean()).to(device)

    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(features, labels),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.AdamW(
        probe.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    for _ in range(epochs):
        probe.train()
        for batch_features, batch_labels in loader:
            logits = probe(batch_features.to(device))
            loss = F.cross_entropy(
                logits,
                batch_labels.to(device),
                weight=class_weights,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return probe


@torch.no_grad()
def predict_probe(probe, features, batch_size=512):
    probe.eval()
    return torch.cat(
        [
            probe(features[start : start + batch_size].to(device)).argmax(dim=1).cpu()
            for start in range(0, len(features), batch_size)
        ]
    )


def evaluate_shape_probe(probe, features, labels, sample_combinations):
    predictions = predict_probe(probe, features)

    per_shape_accuracy = {}
    for shape, shape_id in SHAPE_TO_ID.items():
        mask = labels == shape_id
        if mask.any():
            per_shape_accuracy[shape] = (predictions[mask] == labels[mask]).float().mean().item()

    pair_accuracy = {}
    pair_prediction_counts = {}
    for pair in sorted({combo[:2] for combo in sample_combinations}):
        mask = torch.tensor(
            [combo[:2] == pair for combo in sample_combinations],
            dtype=torch.bool,
        )
        pair_accuracy[pair] = (predictions[mask] == labels[mask]).float().mean().item()
        pair_prediction_counts[pair] = Counter(
            ID_TO_SHAPE[int(prediction)] for prediction in predictions[mask]
        )

    return {
        "accuracy": (predictions == labels).float().mean().item(),
        "balanced_accuracy": sum(per_shape_accuracy.values()) / len(per_shape_accuracy),
        "per_shape_accuracy": per_shape_accuracy,
        "pair_accuracy": pair_accuracy,
        "pair_prediction_counts": pair_prediction_counts,
    }


In [13]:
# The same rendered samples feed both representation stages.
probe_train_features, probe_train_labels, probe_train_combinations = make_probe_feature_sets(
    probe_model,
    train_combinations,
    PROBE_SAMPLES_PER_COMBO,
    PROBE_TRAIN_RENDER_SEED,
)
probe_seen_features, probe_seen_labels, probe_seen_combinations = make_probe_feature_sets(
    probe_model,
    train_combinations,
    PROBE_SAMPLES_PER_COMBO,
    SEEN_EVAL_SEED,
)
probe_held_features, probe_held_labels, probe_held_combinations = make_probe_feature_sets(
    probe_model,
    test_combinations,
    PROBE_SAMPLES_PER_COMBO,
    HELD_OUT_EVAL_SEED,
)

{
    stage: {
        "train_shape": tuple(probe_train_features[stage].shape),
        "seen_shape": tuple(probe_seen_features[stage].shape),
        "held_shape": tuple(probe_held_features[stage].shape),
    }
    for stage in probe_train_features
}


{'pre_projection': {'train_shape': (4200, 4096),
  'seen_shape': (4200, 4096),
  'held_shape': (1200, 4096)},
 'projected': {'train_shape': (4200, 64),
  'seen_shape': (4200, 64),
  'held_shape': (1200, 64)}}

In [14]:
probe_rows = []
probe_details = {}

for stage in ["pre_projection", "projected"]:
    probe = train_linear_probe(
        probe_train_features[stage],
        probe_train_labels,
    )

    train_probe_eval = evaluate_shape_probe(
        probe,
        probe_train_features[stage],
        probe_train_labels,
        probe_train_combinations,
    )
    seen_probe_eval = evaluate_shape_probe(
        probe,
        probe_seen_features[stage],
        probe_seen_labels,
        probe_seen_combinations,
    )
    held_probe_eval = evaluate_shape_probe(
        probe,
        probe_held_features[stage],
        probe_held_labels,
        probe_held_combinations,
    )

    probe_details[stage] = held_probe_eval
    probe_rows.append(
        {
            "stage": stage,
            "train_balanced_accuracy": train_probe_eval["balanced_accuracy"],
            "fresh_seen_balanced_accuracy": seen_probe_eval["balanced_accuracy"],
            "red_triangle_accuracy": held_probe_eval["pair_accuracy"][("red", "triangle")],
            "blue_square_accuracy": held_probe_eval["pair_accuracy"][("blue", "square")],
        }
    )

probe_results = pd.DataFrame(probe_rows)
probe_results


,stage,train_balanced_accuracy,fresh_seen_balanced_accuracy,red_triangle_accuracy,blue_square_accuracy
0,pre_projection,1.0,1.0,0.036667,0.0
1,projected,1.0,1.0,0.545000,0.0


In [15]:
for stage, stage_results in probe_details.items():
    print(f"\n{stage}")
    for pair in sorted(HELD_OUT_PAIRS):
        print(pair, stage_results["pair_prediction_counts"][pair])



pre_projection
('blue', 'square') Counter({'circle': 600})
('red', 'triangle') Counter({'square': 444, 'circle': 134, 'triangle': 22})

projected
('blue', 'square') Counter({'circle': 600})
('red', 'triangle') Counter({'triangle': 327, 'square': 266, 'circle': 7})


### Interpretation rules

Treat the fresh-seen result as a gate: if it is not high, the probe is underfit or otherwise invalid, so held-out results are not interpretable.

- **Pre-projection high, projected high, retrieval low:** square information is linearly accessible throughout the image pathway; the main suspect is cross-modal alignment or text/image similarity geometry.
- **Pre-projection high, projected low:** the CLIP projection/normalization removes or entangles linearly accessible square information.
- **Both low, fresh-seen high:** the frozen visual pathway does not expose held-out blue-square shape linearly under this probe. This supports pair-specific visual entanglement, but it does not prove shape information is absent; it may be nonlinear or the probe may inherit the train split's color-shape correlation.
- **Red-triangle high, blue-square low at the same stage:** the effect is pair-specific, not a general inability to linearly decode shape.
- **Projected high when pre-projection is low:** treat this as a diagnostic warning rather than a clean mechanism claim; optimization, scaling, dimensionality, or normalization may make the probes incomparable.

This single selected seed localizes one clear failure case. A later replication should probe all training seeds before making a population-level claim about where the failure usually occurs.


**Observation**

Held-out green squares and blue squares tend to retrieve same-color circles, while held-out red triangles generalize successfully.

Hypothesis

Square and circle representations are less compositionally separable than triangle representations in the learned joint space.

**Prediction**

If we hold out:

$$ \text{red square}, $$

then:

$$ \text{red circle} $$

should systematically outrank red triangle as the incorrect same-color alternative.

**Falsification**

If red-square retrieval is highly accurate, or if its errors preferentially become red triangle rather than red circle, the simple square/circle hypothesis weakens.